In [1]:
import os, re, json, math, random, gc
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import gensim
from gensim.models import Word2Vec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)

Device: cuda


In [2]:
posts_df = pd.read_csv(r'C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\posts_df.csv')
comments_df = pd.read_csv(r'C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\comments_df.csv')

In [3]:
if 'timestamp' in posts_df.columns:
    posts_df['timestamp'] = pd.to_datetime(posts_df['timestamp'], errors='coerce')
if 'timestamp' in comments_df.columns:
    comments_df['timestamp'] = pd.to_datetime(comments_df['timestamp'], errors='coerce')

print(f"posts_df shape: {posts_df.shape}")
print("Columns:", list(posts_df.columns)[:20])
print("---------------------------")
print(f"comments_df shape: {comments_df.shape}")
print("comments_df:", comments_df.shape, "| columns:", list(comments_df.columns))


posts_df shape: (827, 6)
Columns: ['post_id', 'postUser', 'timestamp', 'likesCount', 'commentsCount', 'keywords_posts']
---------------------------
comments_df shape: (5361, 7)
comments_df: (5361, 7) | columns: ['post_id', 'commentUser', 'timestamp', 'likes', 'polarity', 'sentiment', 'keywords_comments']


In [4]:
def prepare_user_sequences(posts_df, comments_df):
    user_interactions = comments_df.merge(
        posts_df[['post_id', 'keywords_posts']], 
        on='post_id', 
        how='left'
    )
    
    # Sort timestamp
    user_interactions = user_interactions.sort_values('timestamp')
    # Drop columns "likes" 
    user_interactions = user_interactions.drop(columns=['likes'])
    
    return user_interactions

user_interactions = prepare_user_sequences(posts_df, comments_df)
user_interactions.head()

,post_id,commentUser,timestamp,polarity,sentiment,keywords_comments,keywords_posts
5331,822,alantrowe,2021-01-04 10:38:18+00:00,0.2468,neutral,['green_heart'],"['prada', 'fashion', 'fw', 'vogue', 'italia', ..."
5332,823,alantrowe,2021-01-04 10:39:08+00:00,0.0711,neutral,['heart_suit'],"['prada', 'fashion', 'fw', 'vogue', 'italia', ..."
5330,822,thedramatiara,2021-01-05 20:29:21+00:00,-0.0089,neutral,['heart_with_arrow'],"['prada', 'fashion', 'fw', 'vogue', 'italia', ..."
5299,811,wellingthon_lessa,2021-01-08 17:24:27+00:00,-0.0477,neutral,"['desfile', 'em', 'e', 'que', 'duas']","['prada', 'style', 'fashion', 'fw', 'vogue', '..."
5318,816,manzanidap,2021-01-08 18:20:58+00:00,-0.0225,neutral,"['esta', 'es']","['prada', 'style', 'fashion', 'fw', 'vogue', '..."


In [5]:
def build_interaction_sequence(df):
    return df[['post_id', 'timestamp', 'polarity']].to_dict('records')

df_sequence = user_interactions.groupby('commentUser').apply(build_interaction_sequence).reset_index().rename(
    columns={0: 'interaction_sequence'}
)
df_sequence.head()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20444\19145810.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sequence = user_interactions.groupby('commentUser').apply(build_interaction_sequence).reset_index().rename(


,commentUser,interaction_sequence
0,021_alaniss,"[{'post_id': 583, 'timestamp': 2023-06-02 21:1..."
1,100ycientas,"[{'post_id': 319, 'timestamp': 2023-03-29 14:4..."
2,1072.official,"[{'post_id': 65, 'timestamp': 2023-04-07 16:01..."
3,12maaria34,"[{'post_id': 79, 'timestamp': 2023-05-05 21:48..."
4,130wlifestyle,"[{'post_id': 272, 'timestamp': 2023-06-06 01:1..."


In [6]:
df_user_profile = user_interactions.merge(df_sequence, on='commentUser', how='left')
df_user_profile.head()

,post_id,commentUser,timestamp,polarity,sentiment,keywords_comments,keywords_posts,interaction_sequence
0,822,alantrowe,2021-01-04 10:38:18+00:00,0.2468,neutral,['green_heart'],"['prada', 'fashion', 'fw', 'vogue', 'italia', ...","[{'post_id': 822, 'timestamp': 2021-01-04 10:3..."
1,823,alantrowe,2021-01-04 10:39:08+00:00,0.0711,neutral,['heart_suit'],"['prada', 'fashion', 'fw', 'vogue', 'italia', ...","[{'post_id': 822, 'timestamp': 2021-01-04 10:3..."
2,822,thedramatiara,2021-01-05 20:29:21+00:00,-0.0089,neutral,['heart_with_arrow'],"['prada', 'fashion', 'fw', 'vogue', 'italia', ...","[{'post_id': 822, 'timestamp': 2021-01-05 20:2..."
3,811,wellingthon_lessa,2021-01-08 17:24:27+00:00,-0.0477,neutral,"['desfile', 'em', 'e', 'que', 'duas']","['prada', 'style', 'fashion', 'fw', 'vogue', '...","[{'post_id': 811, 'timestamp': 2021-01-08 17:2..."
4,816,manzanidap,2021-01-08 18:20:58+00:00,-0.0225,neutral,"['esta', 'es']","['prada', 'style', 'fashion', 'fw', 'vogue', '...","[{'post_id': 816, 'timestamp': 2021-01-08 18:2..."


In [7]:
df_user_profile.dtypes

post_id                               int64
commentUser                          object
timestamp               datetime64[ns, UTC]
polarity                            float64
sentiment                            object
keywords_comments                    object
keywords_posts                       object
interaction_sequence                 object
dtype: object